# V2 MLP Training (Patched Audio)

Trains two MLP models on V2-patched features (`train_features_v2_patched.pt`).
Input: `z_img` [N,3,512] · `z_aud` [N,128] → `v_teacher` [N,1024].

**Key fixes vs V1:**
- Audio data is now real VGGish (patched from Kaggle) — ~88% coverage
- `z_aud` is L2-normalised per sample in the dataset (raw VGGish L2~1456 vs z_img L2~10.6)
- Input dim 1664 (3×512 + 128), `normalize_output=True` for cosine model
- 50 epochs with cosine LR schedule and 10% val split

## Step 1: Imports & Setup

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 2: Dataset

In [ ]:
class MultimodalDatasetV2(Dataset):
    """
    Loads V2-patched feature files.
    z_img: [N, 3, 512] — three CLIP frames
    z_aud: [N, 128]    — VGGish audio, L2-normalised; silent clips stay as zero vectors
    v_teacher: [N, 1024] — ImageBind multimodal teacher (~unit sphere)
    """
    def __init__(self, file_path):
        data = torch.load(file_path, map_location="cpu", weights_only=False)
        self.z_img     = data["z_img"].float()       # [N, 3, 512]
        self.z_aud     = data["z_aud"].float()       # [N, 128]  raw VGGish [0-255]
        self.v_teacher = data["v_teacher"].float()   # [N, 1024]
        self.has_audio = data.get("has_audio", torch.ones(len(self.z_img), dtype=torch.bool))
        assert len(self.z_img) == len(self.z_aud) == len(self.v_teacher)

        # Safe per-sample L2 normalisation of z_aud.
        # Raw VGGish has L2~1456 vs z_img L2~10.6 — without this audio
        # completely dominates the concatenated input.
        # Silent clips (has_audio=False) are already all-zero and stay zero.
        aud_norms = self.z_aud.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        self.z_aud = self.z_aud / aud_norms
        self.z_aud[~self.has_audio] = 0.0  # ensure silent clips are exactly zero

        print(f"Loaded {len(self.z_img)} samples from {file_path}")
        n_aud = self.has_audio.sum().item()
        print(f"  Audio available: {n_aud}/{len(self.z_img)} ({100*n_aud/len(self.z_img):.1f}%)")

    def __len__(self):
        return len(self.z_img)

    def __getitem__(self, idx):
        return {
            "z_img":     self.z_img[idx],      # [3, 512]
            "z_aud":     self.z_aud[idx],      # [128]
            "v_teacher": self.v_teacher[idx],  # [1024]
            "has_audio": self.has_audio[idx],  # bool scalar
        }

## Step 3: Load Data

In [ ]:
def find_features(filename):
    """Locate a feature file locally or inside a Kaggle input dataset."""
    candidates = [
        filename,
        os.path.join("data_V2_patched", filename),
        os.path.join("..", "data_V2_patched", filename),
        os.path.join("..", "features", "patched_features", filename),
        os.path.join("..", "features", filename),
        os.path.join("/kaggle/working", filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # Kaggle: scan all attached input datasets recursively
    if os.path.exists("/kaggle/input"):
        import glob
        matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"{filename} not found.\n"
        "  On Kaggle : attach the msrvtt-v2-patched dataset as an input.\n"
        "  Locally   : place files in data_V2_patched/."
    )

# On Kaggle, write models to /kaggle/working so they appear as output files
SAVE_DIR = "/kaggle/working/models_v2" if os.path.exists("/kaggle/working") else "models_v2"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model save directory: {SAVE_DIR}")

train_ds = MultimodalDatasetV2(find_features("train_features_v2_patched.pt"))
test_ds  = MultimodalDatasetV2(find_features("test_features_v2_patched.pt"))

# 90/10 train/val split
val_size   = int(0.10 * len(train_ds))
train_size = len(train_ds) - val_size
train_split, val_split = random_split(train_ds, [train_size, val_size],
                                      generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_split, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_split,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,     batch_size=256, shuffle=False, num_workers=2)

print(f"\nSplit: train={train_size}, val={val_size}, test={len(test_ds)}")

## Step 4: Model & Loss Definitions

In [ ]:
class MLPApproximatorV2(nn.Module):
    """
    Flattens 3 CLIP frames → 1536-dim, concatenates VGGish audio → 1664-dim,
    then projects through an MLP to 1024-dim teacher space.
    Optional L2 normalization on output (use for cosine loss to fix magnitude).
    """
    def __init__(self, hidden_dims=(1024, 2048, 1024), output_dim=1024,
                 dropout=0.1, normalize_output=False):
        super().__init__()
        self.normalize_output = normalize_output
        input_dim = 3 * 512 + 128  # 1664
        layers = []
        in_d = input_dim
        for h_d in hidden_dims:
            layers += [
                nn.Linear(in_d, h_d),
                nn.BatchNorm1d(h_d),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            in_d = h_d
        layers.append(nn.Linear(in_d, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, z_img, z_aud):
        # z_img: [B, 3, 512] → flatten → [B, 1536]
        z_img_flat = z_img.reshape(z_img.size(0), -1)
        x = torch.cat([z_img_flat, z_aud], dim=-1)  # [B, 1664]
        out = self.network(x)
        if self.normalize_output:
            out = F.normalize(out, dim=-1)
        return out


class InfoNCELoss(nn.Module):
    """Symmetric InfoNCE (NT-Xent) loss."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, pred, target):
        pred_n   = F.normalize(pred,   dim=-1)
        target_n = F.normalize(target, dim=-1)
        N = pred_n.size(0)
        logits = torch.matmul(pred_n, target_n.T) / self.temperature  # [N, N]
        labels = torch.arange(N, device=pred.device)
        loss_p2t = F.cross_entropy(logits,   labels)
        loss_t2p = F.cross_entropy(logits.T, labels)
        return (loss_p2t + loss_t2p) / 2


def cosine_loss(pred, target):
    return (1 - F.cosine_similarity(pred, target)).mean()


print("Model and loss classes defined.")
print(f"MLP input dim: {3*512+128} (3 frames × 512 + 128 audio)")

## Step 5: Training Utility

In [ ]:
def train_model(model, criterion, train_loader, val_loader, epochs=50, lr=1e-3,
                model_name="model", save_dir="models_v2"):
    os.makedirs(save_dir, exist_ok=True)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    history = {"train": [], "val": []}
    best_val = float("inf")

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss, total_n = 0.0, 0
        for batch in train_loader:
            z_img     = batch["z_img"].to(device)
            z_aud     = batch["z_aud"].to(device)
            v_teacher = batch["v_teacher"].to(device)
            optimizer.zero_grad()
            pred = model(z_img, z_aud)
            loss = criterion(pred, v_teacher)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * z_img.size(0)
            total_n    += z_img.size(0)
        train_loss = total_loss / total_n

        # --- Validate ---
        model.eval()
        val_loss, val_n = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                z_img     = batch["z_img"].to(device)
                z_aud     = batch["z_aud"].to(device)
                v_teacher = batch["v_teacher"].to(device)
                pred = model(z_img, z_aud)
                loss = criterion(pred, v_teacher)
                val_loss += loss.item() * z_img.size(0)
                val_n    += z_img.size(0)
        val_loss /= val_n

        scheduler.step()
        history["train"].append(train_loss)
        history["val"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, f"{model_name}_best.pt"))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | train={train_loss:.4f} | val={val_loss:.4f}"
                  f"{'  *best*' if val_loss == best_val else ''}")
            torch.save(model.state_dict(), os.path.join(save_dir, f"{model_name}_ep{epoch+1}.pt"))

    # Final save
    torch.save(model.state_dict(), os.path.join(save_dir, f"{model_name}_final.pt"))
    print(f"\nTraining complete. Best val loss: {best_val:.4f}")
    print(f"Saved to {save_dir}/{model_name}_best.pt and {model_name}_final.pt")
    return history

## Step 6: Train MLP Cosine (Baseline)

Key improvement: `normalize_output=True` forces the output onto the unit hypersphere, the same space as the ImageBind teacher. This eliminates the MSE=3.28 magnitude issue from V1.

In [ ]:
model_cosine = MLPApproximatorV2(normalize_output=True).to(device)
print(f"MLP Cosine params: {sum(p.numel() for p in model_cosine.parameters()):,}")

history_cosine = train_model(
    model_cosine,
    cosine_loss,
    train_loader, val_loader,
    epochs=50,
    model_name="mlp_cosine_v2",
    save_dir=SAVE_DIR,
)

## Step 7: Train MLP InfoNCE

In [ ]:
model_infonce = MLPApproximatorV2(normalize_output=False).to(device)
print(f"MLP InfoNCE params: {sum(p.numel() for p in model_infonce.parameters()):,}")

history_infonce = train_model(
    model_infonce,
    InfoNCELoss(temperature=0.07),
    train_loader, val_loader,
    epochs=50,
    model_name="mlp_infonce_v2",
    save_dir=SAVE_DIR,
)

## Step 8: Plot Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, history, title in [
    (axes[0], history_cosine,  "MLP Cosine Loss"),
    (axes[1], history_infonce, "MLP InfoNCE Loss"),
]:
    ax.plot(history["train"], label="Train", linewidth=1.5)
    ax.plot(history["val"],   label="Val",   linewidth=1.5, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "mlp_loss_curves.png"), dpi=150)
plt.show()
print(f"Loss curves saved to {SAVE_DIR}/mlp_loss_curves.png")

## Step 9: Quick Test-Set Evaluation

In [ ]:
def quick_eval(model, loader, model_name):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            z_img = batch["z_img"].to(device)
            z_aud = batch["z_aud"].to(device)
            pred  = model(z_img, z_aud)
            preds.append(pred.cpu())
            targets.append(batch["v_teacher"])
    preds   = torch.cat(preds)
    targets = torch.cat(targets)

    pred_n   = F.normalize(preds,   dim=-1)
    target_n = F.normalize(targets, dim=-1)
    cos_sim  = (pred_n * target_n).sum(-1).mean().item()
    mse      = F.mse_loss(preds, targets).item()

    sim_mat = torch.matmul(pred_n, target_n.T)
    N = sim_mat.size(0)
    ranks = [(sim_mat[i] > sim_mat[i, i]).sum().item() + 1 for i in range(N)]
    ranks = torch.tensor(ranks, dtype=torch.float)
    r1  = (ranks <= 1).float().mean().item() * 100
    r5  = (ranks <= 5).float().mean().item() * 100
    r10 = (ranks <= 10).float().mean().item() * 100
    medr = ranks.median().item()

    print(f"\n{model_name}")
    print(f"  Cosine Sim : {cos_sim:.4f}")
    print(f"  MSE        : {mse:.4f}")
    print(f"  R@1/5/10   : {r1:.1f}% / {r5:.1f}% / {r10:.1f}%")
    print(f"  MedR       : {medr:.0f}")
    return {"cosine": cos_sim, "mse": mse, "r1": r1, "r5": r5, "r10": r10, "medr": medr}

for name, m, tag in [
    ("MLP Cosine V2",  model_cosine,  "mlp_cosine_v2"),
    ("MLP InfoNCE V2", model_infonce, "mlp_infonce_v2"),
]:
    best_path = os.path.join(SAVE_DIR, f"{tag}_best.pt")
    if os.path.exists(best_path):
        m.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    quick_eval(m, test_loader, name)